# Week 13 ML

Здесь сделан простой вариант A: классификация дождливого часа по погодным признакам.

Идея такая:
- target: будет ли час дождливым
- признаки: температура, влажность и скорость ветра
- смысл: понять, дает ли даже простая модель что-то лучше, чем baseline


In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use("seaborn-v0_8-whitegrid")

project_root = Path.cwd().resolve()
if not (project_root / "data").exists():
    project_root = project_root.parent

data_path = project_root / "data" / "normalized" / "normalized.csv"
docs_dir = project_root / "docs" / "ml"
docs_dir.mkdir(parents=True, exist_ok=True)

print("Данные:", data_path)
print("Папка для артефактов:", docs_dir)


## Постановка задачи

Target:
- `is_rainy_hour = 1`, если `precipitation > 0`

Features:
- `temperature_2m`
- `relative_humidity_2m`
- `wind_speed_10m`

Что важно по leakage:
- `precipitation` нельзя включать в `X`, потому что target строится именно из нее
- split делается по времени, а не случайно, потому что данные временные


In [ ]:
df = pd.read_csv(data_path, parse_dates=["ts"])
df = df.sort_values("ts").reset_index(drop=True)
df["is_rainy_hour"] = (df["precipitation"] > 0).astype(int)

features = ["temperature_2m", "relative_humidity_2m", "wind_speed_10m"]
target = "is_rainy_hour"

print("Строк всего:", len(df))
print("Период:", df["ts"].min(), "->", df["ts"].max())
print("Распределение target:")
print(df[target].value_counts())


## Train / test split

Деление только по времени:
- train = первые 80% строк
- test = последние 20% строк


In [ ]:
cut = int(len(df) * 0.8)
train = df.iloc[:cut].copy()
test = df.iloc[cut:].copy()

X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Train period:", train["ts"].min(), "->", train["ts"].max())
print("Test period:", test["ts"].min(), "->", test["ts"].max())


## Baseline и модель

Baseline:
- `DummyClassifier(strategy="most_frequent")`

Модель:
- `LogisticRegression`


In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
model.fit(X_train, y_train)
model_pred = model.predict(X_test)


## Метрики


In [ ]:
metrics = []
for name, pred in [("baseline", baseline_pred), ("logreg", model_pred)]:
    metrics.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
    })

metrics_df = pd.DataFrame(metrics)
metrics_df


In [ ]:
plot_df = metrics_df.set_index("model")[["accuracy", "precision", "recall", "f1"]].T

fig, ax = plt.subplots(figsize=(10, 5))
plot_df.plot(kind="bar", ax=ax)
ax.set_title("Сравнение baseline и модели")
ax.set_xlabel("Метрика")
ax.set_ylabel("Значение")
plt.xticks(rotation=0)
fig.tight_layout()

metrics_path = docs_dir / "metrics.png"
fig.savefig(metrics_path, dpi=150, bbox_inches="tight")
plt.show()
print("График сохранен:", metrics_path)


In [ ]:
predictions = test[["ts"] + features + ["precipitation", target]].copy()
predictions["baseline_pred"] = baseline_pred
predictions["model_pred"] = model_pred

sample = predictions.head(30)
sample_path = docs_dir / "predictions_sample.csv"
sample.to_csv(sample_path, index=False)
print("CSV сохранен:", sample_path)
sample.head(10)


## Вывод

Что видно:
- baseline лучше по accuracy, но просто почти всегда выбирает класс `0`
- у baseline `recall = 0` и `f1 = 0`, то есть дождливые часы он вообще не ловит
- логистическая регрессия хуже по accuracy, но сильно лучше по recall и F1

Итог:
- если смотреть только на accuracy, модель не выглядит полезной
- если важно не пропускать дождливые часы, простая модель уже полезнее baseline
- данных мало, поэтому это скорее нормальный первый baseline, а не готовое production-решение
